In [7]:
# https://www.ercot.com/files/docs/2009/05/20/modelingguidelines_v06.pdf

In [39]:
import pandas as pd
from modo_energy_client.modo_energy_api_client import ModoEnergyAPIClient
from datetime import date
import numpy as np
from scipy.stats.mstats import winsorize

In [7]:
client = ModoEnergyAPIClient(cache_requests=True)

In [13]:
prices = client.get_ercot_prices(date(2024, 7, 1), date(2026, 5, 31))

Fetching pages : 26page [00:05,  4.37page/s]


In [59]:
hb_bus_avg_price = prices[prices["settlementPointName"] == "HB_BUSAVG"][
    "settlementPointPrice"
]

In [66]:
hb_bus_avg_price_winsorized = pd.Series(
    winsorize(hb_bus_avg_price, limits=[None, 0.01]), index=hb_bus_avg_price.index
)

In [69]:
daily_ptp = hb_bus_avg_price.groupby(by=lambda d: d.date()).agg(np.ptp)
daily_ptp_winsorized = hb_bus_avg_price_winsorized.groupby(by=lambda d: d.date()).agg(
    np.ptp
)

In [93]:
hb_bus_avg_price_winsorized.groupby(lambda d: d.date()).agg(
    lambda d: d[d.index.hour == 19].item() - d[d.index.hour == 12].item()
)


deliveryDate
2024-07-01    16.00
2024-07-02    27.01
2024-07-03    23.53
2024-07-04    58.57
2024-07-05    13.26
              ...  
2026-05-27    32.18
2026-05-28    83.84
2026-05-29    43.34
2026-05-30    14.36
2026-05-31    28.28
Length: 700, dtype: float64

In [95]:
hb_bus_avg_price_winsorized

deliveryDate
2026-05-31 23:00:00    27.04
2026-05-31 22:00:00    36.64
2026-05-31 21:00:00    43.40
2026-05-31 20:00:00    49.27
2026-05-31 19:00:00    48.79
                       ...  
2024-07-01 04:00:00    16.92
2024-07-01 03:00:00    16.91
2024-07-01 02:00:00    17.21
2024-07-01 01:00:00    18.53
2024-07-01 00:00:00    20.47
Length: 16800, dtype: float64